## download (run only once)

In [ ]:
# from pathlib import Path

# DEST = Path("~/GTEx_data").expanduser()
# DEST.mkdir(exist_ok=True)

# META_BASE = "https://storage.googleapis.com/adult-gtex/annotations/v11/metadata-files"

# meta_files = ["GTEx_Analysis_v11_Annotations_SampleAttributesDS.txt",
#                 "GTEx_Analysis_v11_Annotations_SubjectPhenotypesDS.txt",
#                 "GTEx_Analysis_v11_Annotations_SampleAttributesDD.xlsx",
#                 "GTEx_Analysis_v11_Annotations_SubjectPhenotypesDD.xlsx",
#                 "GTEx_Analysis_v11_Sample_Tissue_Changes_From_v8.txt"
#               ]

# for f in meta_files:
#     url = f"{META_BASE}/{f}"
#     !wget -c -q --show-progress -P {DEST} {url}

In [ ]:
# MAIN_BASE = "https://storage.googleapis.com/adult-gtex/bulk-gex/v11/rna-seq"
# main_files = ["GTEx_Analysis_2026-05-19_v11_RNASeQCv2.4.3_gene_reads.parquet",
#               "GTEx_Analysis_2026-05-19_v11_RNASeQCv2.4.3_gene_tpm.parquet"]

# for f in main_files:
#     url = f"{MAIN_BASE}/{f}"
#     !wget -c -q --show-progress -P {DEST} {url}

In [ ]:
# !wget -c -q --show-progress -P {DEST} https://storage.googleapis.com/adult-gtex/references/v11/reference-tables/gencode.v47.genes.gtf

## main processing

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
DATA_PATH = "~/GTEx_data/GTEx_Analysis_2026-05-19_v11_RNASeQCv2.4.3_gene_tpm.parquet"

In [ ]:
gtex_tpm_pq = pd.read_parquet(DATA_PATH)

In [ ]:
gtex_samp_ids = [v for v in gtex_tpm_pq.columns if v.startswith('GTEX-')]
len(gtex_samp_ids) == len(set(gtex_samp_ids))

In [ ]:
metadata_path = "~/GTEx_data/GTEx_Analysis_v11_Annotations_SampleAttributesDS.txt"
gtex_meta_df_all = pd.read_csv(metadata_path,sep="\t")

In [ ]:
meta_samp_ids = [v for v in gtex_meta_df_all['SAMPID'] if v.startswith('GTEX-')]

In [ ]:
set(gtex_samp_ids) <= set(meta_samp_ids)

In [ ]:
gtex_meta_df = gtex_meta_df_all[gtex_meta_df_all['SAMPID'].isin(gtex_samp_ids)]

In [ ]:
(gtex_meta_df['SMTSD'].value_counts()>100).value_counts()

In [ ]:
broad_fine_tissue_counts = gtex_meta_df.groupby('SMTS')['SMTSD'].value_counts()

In [ ]:
num_fine_per_broad = (broad_fine_tissue_counts>100).groupby('SMTS').sum()

In [ ]:
num_fine_per_broad[num_fine_per_broad>2]

In [ ]:
broad_fine_tissue_counts.loc[num_fine_per_broad[num_fine_per_broad>2].index]

In [ ]:
# # Sample annotations: one row per sample; SMTSD = detailed tissue name
# attrs = pd.read_csv(, sep="\t")
# print(attrs["SMTSD"].value_counts().head(10))

### restrict to brain (for now)

In [ ]:
brain_meta_df = gtex_meta_df[gtex_meta_df['SMTS']=='Brain'].copy()
brain_meta_df.columns

In [ ]:
brain_meta_df['SMTSD_SAMPID'] = brain_meta_df['SMTSD'].str.replace(' ','') + '_' + brain_meta_df['SAMPID']

In [ ]:
sampid_tissue_dict= brain_meta_df.set_index('SAMPID')['SMTSD_SAMPID'].to_dict()

In [ ]:
gtex_tpm_pq_brain = gtex_tpm_pq[brain_meta_df['SAMPID']].copy()

In [ ]:
gtex_tpm_pq_brain

In [ ]:
gtex_count_pq = pd.read_parquet("~/GTEx_data/GTEx_Analysis_2026-05-19_v11_RNASeQCv2.4.3_gene_reads.parquet")

In [ ]:
gtex_count_pq_brain = gtex_count_pq[brain_meta_df['SAMPID']].copy()

In [ ]:
del(gtex_count_pq)

In [ ]:
assert gtex_tpm_pq_brain.shape == gtex_count_pq_brain.shape

### see gene reference

In [ ]:
gtf = pd.read_csv("~/GTEx_data/gencode.v47.genes.gtf", sep="\t", comment="#", header=None,
                  usecols=[0, 2, 8], names=["chrom", "feature", "attrs"])
genes = gtf[gtf["feature"] == "gene"]
genes = genes.assign(
    gene_id=genes["attrs"].str.extract(r'gene_id "([^"]+)"', expand=False),
    gene_type=genes["attrs"].str.extract(r'gene_type "([^"]+)"', expand=False),
    gene_name=genes["attrs"].str.extract(r'gene_name "([^"]+)"', expand=False),
).set_index("gene_id")

In [ ]:
genes

### filter genes

- TPM filter, filter by reads
- examine types using reference
- extract highly variable

In [ ]:
# [v for v in genes['gene_type'].unique() if 'pseudogene' not in v]

In [ ]:
def get_topK_dataframe(subdf_ids,gtex_tpm_pq,gtex_count_pq,gene_ref,tissue_name='Brain - Amygdala',n_genes=1000,gene_bio_filter=True):
    
    # subdf_ids = meta_df[meta_df['SMTSD']==tissue_name]['SAMPID']
    n_top = n_genes

    gtex_tpm_subdf = gtex_tpm_pq[subdf_ids]
    gtex_count_subdf = gtex_count_pq[subdf_ids]

    tpm_passing_genes = (gtex_tpm_subdf>0.1).mean(axis=1)>=0.2
    count_passing_genes = (gtex_count_subdf>=6).mean(axis=1)>=0.2

    gene_filter_lst = tpm_passing_genes&count_passing_genes
    
    if gene_bio_filter:
        gene_type_lst = ['protein_coding','lncRNA']        
        gene_gtex_sub = gene_ref.loc[gene_filter_lst]
        bio_passing_genes = gene_gtex_sub['gene_type'].isin(gene_type_lst)
        gene_filter_lst = bio_passing_genes&gene_filter_lst
        
    gtex_tpm_subdf_filt = gtex_tpm_subdf.loc[gene_filter_lst].copy()

    filtered_log_df = np.log1p(gtex_tpm_subdf_filt)

    filtered_log_df['gene_var'] = filtered_log_df.var(axis=1)

    filtered_log_df = filtered_log_df.reset_index().set_index(['Name','gene_var'])

    sorted_df = filtered_log_df.sort_values(by='gene_var',ascending=False)

    return sorted_df[:n_top].T

In [ ]:
topK_gene_df = get_topK_dataframe(brain_meta_df,gtex_tpm_pq_brain,gtex_count_pq_brain,genes,tissue_name='Brain - Amygdala',n_genes=1000,gene_bio_filter=True)

In [ ]:
brain_meta_df[brain_meta_df['SMTSD']==t]['SAMPID']

In [ ]:
brain_meta_df["SMTSD"].value_counts().min()

In [ ]:
import re
from pathlib import Path
import os

In [ ]:
n_sample, n_gene = 100, 100
tissues = brain_meta_df["SMTSD"].dropna().unique()
num_seeds = 10
smts_val = "Brain"

OUT = Path(os.environ["EPHEMERAL"]) / "pyspoc_GTEx_data" / smts_val 
OUT.mkdir(parents=True, exist_ok=True)

print(brain_meta_df["SMTSD"].value_counts().min(),n_sample)
assert brain_meta_df["SMTSD"].value_counts().min() > n_sample

rows = []
for ix,t in enumerate(tissues):
    
    t_clean = re.sub(r'[^A-Za-z0-9]+', '_', t).strip('_').lower()
    TISSUE_OUT = OUT / f"{t_clean}" / f"N{n_sample}_P{n_gene}"
    META_OUT = TISSUE_OUT / "metadata"
    META_OUT.mkdir(parents=True, exist_ok=True)

    for seed in range(num_seeds):
        
        seed_val = seed        
        ids = brain_meta_df.loc[brain_meta_df["SMTSD"] == t, "SAMPID"]
        sub_ids = ids.sample(n=n_sample, replace=False, random_state=seed_val).tolist()
            
        topK_gene_df = get_topK_dataframe(sub_ids,gtex_tpm_pq_brain,gtex_count_pq_brain,
                                        genes,tissue_name=t,n_genes=n_gene,gene_bio_filter=True)
        
        N = topK_gene_df.shape[0]
        P = topK_gene_df.shape[1]
        sid = f"gtex_{t_clean}_N{N}_P{P}_seed{seed_val}"

        npy_save_out = TISSUE_OUT / f"{sid}.npy"
        np.save(npy_save_out, topK_gene_df.to_numpy(dtype=float))
        pd.Series(topK_gene_df.index).to_csv(META_OUT / f"{sid}_samples.csv", index=False, header=False)
        pd.DataFrame(topK_gene_df.columns).to_csv(META_OUT / f"{sid}_genes.csv", index=False, header=False)

        rows.append({"safe_data_id": sid, "path": str(npy_save_out),
                    "SMTSD": t,"SMTS" : smts_val , "N": N, "P": P,"seed":seed_val,"metadata_dir":str(META_OUT)})

manifest = pd.DataFrame(rows)

In [ ]:
# manifest

In [ ]:
assert manifest["safe_data_id"].is_unique
manifest.to_csv(OUT / f"gtex_N{n_sample}_P{n_gene}.csv", index=False)

In [ ]:
homepath = Path('~/hcda_pyspoc_project/py-spoc/working_analyses/hjResearch_2025/hpc_pyspoc_scripts/metadata_files')
manifest.to_csv(homepath / f"gtex_N{n_sample}_P{n_gene}.csv", index=False)

In [ ]:
# (tpm_passing_genes&count_passing_genes).value_counts()

In [ ]:
# gtex_tpm_subdf_filt.shape